# 03 - Ensemble Stacking and Weighted Voting
Objetivo: combinar XGBoost + Random Forest con Voting y Stacking para mejorar Directional Accuracy y robustez en H2/H3 bajo politica de senal pura.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, VotingRegressor, VotingClassifier, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / 'src').exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings('ignore')

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / 'data' / 'processed' / 'dataset_entrenamiento_final.csv'
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError('dataset_entrenamiento_final.csv not found under data/processed')

df = pd.read_csv(dataset_path, parse_dates=['date'])
print('dataset:', dataset_path.resolve())
print('shape:', df.shape)

blacklist_total = [
    'precio_provincial_lag_1',
    'precio_provincial_lag_2',
    'precio_provincial_lag_3',
    'precio_vecinos_media_lag1',
]

target_candidates = ['precio_provincial_TARGET_H1', 'precio_provincial_TARGET_H2', 'precio_provincial_TARGET_H3']
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f'Missing target columns: {missing_targets}')

split_date = pd.Timestamp('2021-01-01')
train_mask = df['date'] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print('train rows:', train_df.shape[0], 'test rows:', test_df.shape[0])

identifiers = ['date', 'provincia', 'cereal_predominante']
training_cols = get_training_features(df)
feature_cols = [c for c in training_cols if c in df.columns and c not in identifiers + target_candidates]
feature_cols = [c for c in feature_cols if c not in blacklist_total]

leaks = [c for c in blacklist_total if c in feature_cols]
if leaks:
    raise ValueError(f'Leakage detected in feature_cols: {leaks}')

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = 'precio_provincial_lag_1'
if base_price_col not in df.columns:
    raise ValueError('precio_provincial_lag_1 missing for target construction')

horizons = [1, 2, 3]
crisis_start = pd.Timestamp('2022-01-01')
crisis_end = pd.Timestamp('2022-06-30')

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


In [2]:
def build_targets(horizon: int):
    target_reg = f'precio_provincial_TARGET_H{horizon}'
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    return y_train_reg, y_test_reg, base_train, base_test

def make_return_targets(y_reg, base):
    return (y_reg - base) / base

def make_direction_targets(y_reg, base):
    return (y_reg > base).astype(int)

def regression_metrics(y_true, y_pred):
    aligned = pd.concat([y_true, y_pred], axis=1).dropna()
    if aligned.empty:
        return {'MAE': np.nan, 'RMSE': np.nan, 'Pearson': np.nan}
    y_t = aligned.iloc[:, 0]
    y_p = aligned.iloc[:, 1]
    pear = pearsonr(y_t, y_p)[0] if y_t.nunique() > 1 else np.nan
    return {
        'MAE': float(mean_absolute_error(y_t, y_p)),
        'RMSE': float(np.sqrt(mean_squared_error(y_t, y_p))),
        'Pearson': float(pear) if pear == pear else np.nan,
    }

def directional_accuracy_from_returns(y_true_ret, y_pred_ret):
    aligned = pd.concat([y_true_ret, y_pred_ret], axis=1).dropna()
    if aligned.empty:
        return np.nan
    sign_true = np.sign(aligned.iloc[:, 0].values)
    sign_pred = np.sign(aligned.iloc[:, 1].values)
    return float((sign_true == sign_pred).mean())

def anti_crisis_mae(y_true_ret, y_pred_ret, y_test_index):
    crisis_idx = y_test_index[(test_df.loc[y_test_index, 'date'] >= crisis_start) & (test_df.loc[y_test_index, 'date'] <= crisis_end)]
    if len(crisis_idx) == 0:
        return np.nan
    return float(mean_absolute_error(y_true_ret.loc[crisis_idx], y_pred_ret.loc[crisis_idx]))

def make_reg_estimators():
    xgb = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective='reg:squarederror',
        n_estimators=400,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.9,
        reg_lambda=3.0,
        reg_alpha=0.5,
    )
    rf = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=2,
        max_features='sqrt',
    )
    return xgb, rf

def make_clf_estimators():
    xgbc = XGBClassifier(
        random_state=42,
        n_jobs=-1,
        objective='binary:logistic',
        n_estimators=350,
        learning_rate=0.05,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.9,
        reg_lambda=3.0,
        reg_alpha=0.5,
        eval_metric='logloss',
    )
    rfc = RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=2,
        max_features='sqrt',
    )
    return xgbc, rfc

## 1) Torneo de Votacion Ponderada
Se optimizan pesos para VotingRegressor en CV temporal con objetivo MAE sobre tramos de alta volatilidad (proxy anti-crisis). En clasificacion se optimiza DA en CV.

In [3]:
weight_grid = [(0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (0.8, 0.2), (0.9, 0.1)]
tscv = TimeSeriesSplit(n_splits=5)

voting_weight_rows = []
results_rows = []

for h in horizons:
    y_train_reg, y_test_reg, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()

    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    y_train_ret = make_return_targets(y_train_reg.loc[train_mask_h], base_train_h)
    y_test_ret = make_return_targets(y_test_reg.loc[test_mask_h], base_test_h)
    y_train_dir = make_direction_targets(y_train_reg.loc[train_mask_h], base_train_h)
    y_test_dir = make_direction_targets(y_test_reg.loc[test_mask_h], base_test_h)

    best_w_reg = None
    best_cv_mae = np.inf

    for w_xgb, w_rf in weight_grid:
        fold_scores = []
        for tr_idx, va_idx in tscv.split(X_train_h):
            X_tr, X_va = X_train_h.iloc[tr_idx], X_train_h.iloc[va_idx]
            y_tr, y_va = y_train_ret.iloc[tr_idx], y_train_ret.iloc[va_idx]

            xgb, rf = make_reg_estimators()
            vote_reg = VotingRegressor(estimators=[('xgb', xgb), ('rf', rf)], weights=[w_xgb, w_rf])
            vote_reg.fit(X_tr, y_tr)
            pred_va = pd.Series(vote_reg.predict(X_va), index=y_va.index)

            q = y_va.abs().quantile(0.8)
            stress_idx = y_va.index[y_va.abs() >= q]
            score = mean_absolute_error(y_va.loc[stress_idx], pred_va.loc[stress_idx]) if len(stress_idx) > 0 else mean_absolute_error(y_va, pred_va)
            fold_scores.append(score)

        cv_mae = float(np.mean(fold_scores))
        if cv_mae < best_cv_mae:
            best_cv_mae = cv_mae
            best_w_reg = (w_xgb, w_rf)

    best_w_clf = None
    best_cv_da = -np.inf

    for w_xgb, w_rf in weight_grid:
        fold_da = []
        for tr_idx, va_idx in tscv.split(X_train_h):
            X_tr, X_va = X_train_h.iloc[tr_idx], X_train_h.iloc[va_idx]
            y_tr, y_va = y_train_dir.iloc[tr_idx], y_train_dir.iloc[va_idx]

            xgbc, rfc = make_clf_estimators()
            vote_clf = VotingClassifier(estimators=[('xgbc', xgbc), ('rfc', rfc)], voting='soft', weights=[w_xgb, w_rf])
            vote_clf.fit(X_tr, y_tr)
            pred_va = vote_clf.predict(X_va)
            fold_da.append(accuracy_score(y_va, pred_va))

        cv_da = float(np.mean(fold_da))
        if cv_da > best_cv_da:
            best_cv_da = cv_da
            best_w_clf = (w_xgb, w_rf)

    xgb, rf = make_reg_estimators()
    reg_xgb = xgb.fit(X_train_h, y_train_ret)
    reg_rf = rf.fit(X_train_h, y_train_ret)

    pred_xgb = pd.Series(reg_xgb.predict(X_test_h), index=y_test_ret.index)
    pred_rf = pd.Series(reg_rf.predict(X_test_h), index=y_test_ret.index)

    vote_reg = VotingRegressor(estimators=[('xgb', make_reg_estimators()[0]), ('rf', make_reg_estimators()[1])], weights=list(best_w_reg))
    vote_reg.fit(X_train_h, y_train_ret)
    pred_vote = pd.Series(vote_reg.predict(X_test_h), index=y_test_ret.index)

    xgbc, rfc = make_clf_estimators()
    clf_xgb = xgbc.fit(X_train_h, y_train_dir)
    clf_rf = rfc.fit(X_train_h, y_train_dir)

    pred_dir_xgb = pd.Series(clf_xgb.predict(X_test_h), index=y_test_dir.index)
    pred_dir_rf = pd.Series(clf_rf.predict(X_test_h), index=y_test_dir.index)

    vote_clf = VotingClassifier(estimators=[('xgbc', make_clf_estimators()[0]), ('rfc', make_clf_estimators()[1])], voting='soft', weights=list(best_w_clf))
    vote_clf.fit(X_train_h, y_train_dir)
    pred_dir_vote = pd.Series(vote_clf.predict(X_test_h), index=y_test_dir.index)

    voting_weight_rows.append({
        'horizon': h,
        'w_reg_xgb': best_w_reg[0],
        'w_reg_rf': best_w_reg[1],
        'cv_stress_mae_reg': best_cv_mae,
        'w_clf_xgb': best_w_clf[0],
        'w_clf_rf': best_w_clf[1],
        'cv_da_clf': best_cv_da,
    })

    reg_models = {
        'XGB_reg': pred_xgb,
        'RF_reg': pred_rf,
        'Voting_reg': pred_vote,
    }
    for model_name, pred in reg_models.items():
        m = regression_metrics(y_test_ret, pred)
        results_rows.append({
            'horizon': h,
            'model': model_name,
            'type': 'regression',
            'MAE': m['MAE'],
            'RMSE': m['RMSE'],
            'Pearson': m['Pearson'],
            'DA': directional_accuracy_from_returns(y_test_ret, pred),
            'MAE_crisis': anti_crisis_mae(y_test_ret, pred, y_test_ret.index),
        })

    clf_models = {
        'XGB_clf': pred_dir_xgb,
        'RF_clf': pred_dir_rf,
        'Voting_clf': pred_dir_vote,
    }
    for model_name, pred_cls in clf_models.items():
        results_rows.append({
            'horizon': h,
            'model': model_name,
            'type': 'classification',
            'MAE': np.nan,
            'RMSE': np.nan,
            'Pearson': np.nan,
            'DA': float(accuracy_score(y_test_dir, pred_cls)),
            'MAE_crisis': np.nan,
        })

weights_df = pd.DataFrame(voting_weight_rows)
results_df = pd.DataFrame(results_rows)

display(weights_df)
results_df.sort_values(['horizon', 'type', 'model'])

,horizon,w_reg_xgb,w_reg_rf,cv_stress_mae_reg,w_clf_xgb,w_clf_rf,cv_da_clf
0,1,0.9,0.1,0.083791,0.9,0.1,0.561290
1,2,0.7,0.3,0.106500,0.5,0.5,0.565962
2,3,0.9,0.1,0.120512,0.5,0.5,0.616463


,horizon,model,type,MAE,RMSE,Pearson,DA,MAE_crisis
4,1,RF_clf,classification,NaN,NaN,NaN,0.644283,NaN
5,1,Voting_clf,classification,NaN,NaN,NaN,0.617060,NaN
3,1,XGB_clf,classification,NaN,NaN,NaN,0.617060,NaN
1,1,RF_reg,regression,0.047269,0.067789,0.529458,0.692075,0.117943
2,1,Voting_reg,regression,0.046403,0.065053,0.570935,0.679371,0.103509
0,1,XGB_reg,regression,0.046817,0.065215,0.548727,0.671506,0.102039
10,2,RF_clf,classification,NaN,NaN,NaN,0.684113,NaN
11,2,Voting_clf,classification,NaN,NaN,NaN,0.647167,NaN
9,2,XGB_clf,classification,NaN,NaN,NaN,0.642857,NaN
7,2,RF_reg,regression,0.060493,0.090658,0.631877,0.750616,0.166923


## 2) Stacking con Ridge como Meta-Learner
StackingRegressor con XGB+RF como base y Ridge(alpha alto) para combinar predicciones en escenarios de distinto regimen.

In [6]:
stacking_rows = []



for h in horizons:

    y_train_reg, y_test_reg, base_train, base_test = build_targets(h)

    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()

    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()



    X_train_h = X_train.loc[train_mask_h]

    X_test_h = X_test.loc[test_mask_h]

    base_train_h = base_train.loc[train_mask_h]

    base_test_h = base_test.loc[test_mask_h]



    y_train_ret = make_return_targets(y_train_reg.loc[train_mask_h], base_train_h)

    y_test_ret = make_return_targets(y_test_reg.loc[test_mask_h], base_test_h)



    xgb, rf = make_reg_estimators()

    stack = StackingRegressor(

        estimators=[('xgb', xgb), ('rf', rf)],

        final_estimator=Ridge(alpha=100.0),

        passthrough=False,

        cv=5,

        n_jobs=-1,

    )

    stack.fit(X_train_h, y_train_ret)

    pred_stack = pd.Series(stack.predict(X_test_h), index=y_test_ret.index)



    m = regression_metrics(y_test_ret, pred_stack)

    stacking_rows.append({

        'horizon': h,

        'model': 'Stacking_reg',

        'type': 'regression',

        'MAE': m['MAE'],

        'RMSE': m['RMSE'],

        'Pearson': m['Pearson'],

        'DA': directional_accuracy_from_returns(y_test_ret, pred_stack),

        'MAE_crisis': anti_crisis_mae(y_test_ret, pred_stack, y_test_ret.index),

    })



stacking_df = pd.DataFrame(stacking_rows)

final_results = pd.concat([results_df, stacking_df], ignore_index=True)



comparison_h23 = final_results[final_results['horizon'].isin([2, 3])].copy()

comparison_h23.sort_values(['horizon', 'type', 'model'])

,horizon,model,type,MAE,RMSE,Pearson,DA,MAE_crisis
10,2,RF_clf,classification,NaN,NaN,NaN,0.684113,NaN
11,2,Voting_clf,classification,NaN,NaN,NaN,0.647167,NaN
9,2,XGB_clf,classification,NaN,NaN,NaN,0.642857,NaN
7,2,RF_reg,regression,0.060493,0.090658,0.631877,0.750616,0.166923
19,2,Stacking_reg,regression,0.070872,0.092865,0.658179,0.692118,0.140948
8,2,Voting_reg,regression,0.063421,0.093131,0.689779,0.644089,0.160225
6,2,XGB_reg,regression,0.066564,0.095263,0.648296,0.629310,0.157609
16,3,RF_clf,classification,NaN,NaN,NaN,0.593103,NaN
17,3,Voting_clf,classification,NaN,NaN,NaN,0.647649,NaN
15,3,XGB_clf,classification,NaN,NaN,NaN,0.645768,NaN


In [7]:
summary_h23 = comparison_h23.pivot_table(
    index=['horizon', 'model', 'type'],
    values=['Pearson', 'DA', 'MAE', 'MAE_crisis'],
    aggfunc='first',
).reset_index()

best_h23 = summary_h23.sort_values(['horizon', 'DA', 'Pearson'], ascending=[True, False, False]).groupby('horizon').head(3)

display(summary_h23)
best_h23

,horizon,model,type,DA,MAE,MAE_crisis,Pearson
0,2,RF_clf,classification,0.684113,NaN,NaN,NaN
1,2,RF_reg,regression,0.750616,0.060493,0.166923,0.631877
2,2,Stacking_reg,regression,0.692118,0.070872,0.140948,0.658179
3,2,Voting_clf,classification,0.647167,NaN,NaN,NaN
4,2,Voting_reg,regression,0.644089,0.063421,0.160225,0.689779
5,2,XGB_clf,classification,0.642857,NaN,NaN,NaN
6,2,XGB_reg,regression,0.629310,0.066564,0.157609,0.648296
7,3,RF_clf,classification,0.593103,NaN,NaN,NaN
8,3,RF_reg,regression,0.719122,0.073381,0.187942,0.665168
9,3,Stacking_reg,regression,0.785580,0.087232,0.165579,0.509688


,horizon,model,type,DA,MAE,MAE_crisis,Pearson
1,2,RF_reg,regression,0.750616,0.060493,0.166923,0.631877
2,2,Stacking_reg,regression,0.692118,0.070872,0.140948,0.658179
0,2,RF_clf,classification,0.684113,NaN,NaN,NaN
9,3,Stacking_reg,regression,0.785580,0.087232,0.165579,0.509688
8,3,RF_reg,regression,0.719122,0.073381,0.187942,0.665168
10,3,Voting_clf,classification,0.647649,NaN,NaN,NaN


## 3) Reporte de certificacion de ensamble
Exporta resultados, comparativa H2/H3 y pesos finales recomendados para produccion.

In [ ]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / 'reports'
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / 'reports'
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / 'CERTIFICACION_ENSAMBLE_V1.md'

prod_weights = weights_df[weights_df['horizon'].isin([2, 3])].copy()

lines = [
    '# CERTIFICACION_ENSAMBLE_V1',
    '',
    '## Configuracion de senal pura',
    'Se excluyeron del entrenamiento: precio_provincial_lag_1/2/3 y precio_vecinos_media_lag1.',
    '',
    '## Pesos optimizados Voting (produccion)',
    prod_weights.to_markdown(index=False),
    '',
    '## Resultados completos ensamble/base',
    final_results.sort_values(['horizon', 'type', 'model']).to_markdown(index=False),
    '',
    '## Comparativa foco H2 y H3',
    summary_h23.sort_values(['horizon', 'type', 'DA'], ascending=[True, True, False]).to_markdown(index=False),
    '',
    '## Recomendacion de despliegue',
    '- Usar Voting con pesos por horizonte para estabilidad de MAE anti-crisis.',
    '- Mantener StackingRegressor como challenger en monitoreo continuo para H3.',
    '- Priorizar modelos con mejor DA en H2/H3 cuando haya empate de Pearson.',
]

report_path.write_text('\n'.join(lines), encoding='utf-8')
print('Reporte guardado en:', report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\reports\CERTIFICACION_ENSAMBLE_V1.md


: 